# Taller: Red Neuronal Artificial (ANN) - Clasificación Multiclase
Objetivo: Predecir el Credit Score (0 = Malo, 1 = Estándar, 2 = Bueno)


## 1. Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {keras.__version__}')
print('Librerías cargadas correctamente')

In [ ]:
#dame la version de python
import sys
print(f'Python      : {sys.version}')


## 2. Carga de Datos

In [ ]:
#Cargar el dataset desde Google Drive
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_excel('/content/drive/MyDrive/riesgo.xlsx')

print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head()

In [ ]:
print('Tipos de datos:')
print(df.dtypes)
print()
print('Valores nulos por columna:')
print(df.isnull().sum())

In [ ]:
print('Estadísticas descriptivas:')
df.describe().T

## 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Distribución de la variable objetivo
etiquetas = {0: 'Malo (0)', 1: 'Estándar (1)', 2: 'Bueno (2)'}
colores   = ['#e74c3c', '#3498db', '#2ecc71']
conteo    = df['Credit_Score'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribución de Credit Score (Variable Objetivo)', fontsize=14, fontweight='bold')

axes[0].bar([etiquetas[k] for k in conteo.index], conteo.values, color=colores, edgecolor='black')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Cantidad de registros')
axes[0].set_title('Conteo por clase')
for i, v in enumerate(conteo.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(conteo.values, labels=[etiquetas[k] for k in conteo.index],
            autopct='%1.1f%%', colors=colores, startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporción por clase')

plt.tight_layout()
plt.show()
print('Dataset desbalanceado: Estándar (1) domina con 49%. Se corregirá con class_weight.')

In [ ]:
# Boxplots para comparar variables numéricas clave entre clases
cols_num = ['Age', 'Annual_Income', 'Interest_Rate', 'Outstanding_Debt',
            'Credit_Utilization_Ratio', 'Monthly_Balance']

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('Boxplots por Credit Score', fontsize=14, fontweight='bold')

for ax, col in zip(axes.flatten(), cols_num):
    datos_box = [df[df['Credit_Score'] == s][col].dropna().values for s in [0, 1, 2]]
    bp = ax.boxplot(datos_box, patch_artist=True,
                    medianprops={'color': 'black', 'linewidth': 2})
    for patch, color in zip(bp['boxes'], colores):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticklabels(['Malo', 'Estándar', 'Bueno'])
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
# Mapa de calor de correlaciones
cols_corr = ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
             'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date',
             'Num_of_Delayed_Payment', 'Outstanding_Debt', 'Credit_Utilization_Ratio',
             'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly',
             'Monthly_Balance', 'Credit_Score']

fig, ax = plt.subplots(figsize=(14, 11))
corr = df[cols_corr].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlación (variables numéricas)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Preprocesamiento

# Decisiones clave:
- Type_of_Loan se elimina: tiene 6260 valores únicos (combinaciones de texto libre). Codificarla con LabelEncoder introduciría un orden artificial que no existe.
- Payment_of_Min_Amount 'NM'** → se imputa como 'No' (sin pago mínimo registrado).

In [ ]:
# Eliminamos identificadores y Type_of_Loan (alta cardinalidad)
cols_drop = ['Customer_ID', 'Name', 'SSN', 'Type_of_Loan']
df_clean  = df.drop(columns=cols_drop).copy()

# Corregir valor 'NM' en Payment_of_Min_Amount
df_clean['Payment_of_Min_Amount'] = df_clean['Payment_of_Min_Amount'].replace('NM', 'No')

print(f'Columnas eliminadas : {cols_drop}')
print(f'Dimensiones finales : {df_clean.shape}')
print(f'Nulos restantes     : {df_clean.isnull().sum().sum()}')

In [ ]:
# Separar features (X) y variable objetivo (y)
X = df_clean.drop(columns=['Credit_Score'])
y = df_clean['Credit_Score']

# Identificar columnas categóricas y numéricas
cols_cat = X.select_dtypes(include='object').columns.tolist()
cols_num = X.select_dtypes(exclude='object').columns.tolist()

print(f'Features totales    : {X.shape[1]}')
print(f'Categóricas ({len(cols_cat)}): {cols_cat}')
print(f'Numéricas   ({len(cols_num)}): {cols_num}')

In [ ]:
# LabelEncoder: convierte texto a enteros para que la red pueda procesarlos
X_enc = X.copy()
for col in cols_cat:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X_enc[col].astype(str))
    print(f'  {col}: {list(le.classes_)} → {list(range(len(le.classes_)))}')

print('\nCodificación completada')

In [ ]:
# División estratificada: 70% Train | 15% Validación | 15% Test
# La división se hace ANTES de escalar para evitar data leakage

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_enc, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.1765,  # ~15% del total
    random_state=42, stratify=y_train_val
)

print(f'Train      : {X_train.shape[0]} muestras  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Validación : {X_val.shape[0]} muestras  ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'Test       : {X_test.shape[0]} muestras  ({X_test.shape[0]/len(X)*100:.1f}%)')

In [ ]:
# StandardScaler: ajuste solo sobre train, luego transformar val y test
# Esto evita que información del test "contamine" el entrenamiento
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print(f'Media  train (primeras 5): {X_train_sc[:,:5].mean(axis=0).round(4)}')
print(f'Desv.  train (primeras 5): {X_train_sc[:,:5].std(axis=0).round(4)}')
print('Normalización completada')

In [ ]:
# One-Hot Encoding de las etiquetas
# Necesario porque la capa de salida tiene 3 neuronas con activación Softmax
y_train_oh = to_categorical(y_train, num_classes=3)
y_val_oh   = to_categorical(y_val,   num_classes=3)
y_test_oh  = to_categorical(y_test,  num_classes=3)

print(f'Forma one-hot train : {y_train_oh.shape}')

## 5. Manejo del Desbalance con class_weight

Uso de class_weight directamente en model.fit().  
Esto le indica a Keras que penalice más los errores cometidos en las clases con menos muestras, compensando el desbalance durante el entrenamiento sin modificar el dataset.

In [ ]:
# compute_class_weight calcula automáticamente el peso inverso a la frecuencia de cada clase
pesos = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=y_train.values
)
class_weight_dict = {0: pesos[0], 1: pesos[1], 2: pesos[2]}

print('Pesos por clase:')
for clase, peso in class_weight_dict.items():
    print(f'  Clase {clase} ({etiquetas[clase]}): {peso:.4f}')

print()

Interpretación: la clase Bueno (2) recibe mayor peso porque tiene
menos muestras → sus errores penalizan más durante el entrenamiento.

## 6. Construcción de la Red Neuronal

Arquitectura: **4 capas ocultas** (64 → 32 → 16 ) + capa de salida Softmax.  
Más neuronas en la primera capa permite capturar relaciones más complejas entre las 21 features.

In [ ]:
n_features = X_train_sc.shape[1]

model = Sequential([
    Input(shape=(n_features,)),
    Dense(64, activation='relu'),
    Dense(32,  activation='relu'),
    Dense(16,  activation='relu'),
    Dense(3,   activation='softmax')   
])

model.summary()

In [ ]:
# Compilación del modelo, optimizer Adam con learning_rate=0.001 (valor estándar), loss categorical_crossentropy: función de pérdida para multiclase con one-hot, metrics accuracy: porcentaje de predicciones correctas
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('Modelo compilado y listo para entrenar')

## 7. Entrenamiento

Se usa EarlyStopping con restore_best_weights=True para detener el entrenamiento
cuando val_loss deja de mejorar y conservar los mejores pesos automáticamente.

El parámetro class_weight se pasa directamente a model.fit() para compensar el desbalance.

In [ ]:
# EarlyStopping: detiene si val_loss no mejora en 20 épocas consecutivas
# restore_best_weights=True: recupera los pesos del mejor momento automáticamente
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Entrenamiento con class_weight para compensar el desbalance de clases
history = model.fit(
    X_train_sc, y_train_oh,
    validation_data=(X_val_sc, y_val_oh),
    epochs=1000,
    batch_size=64,
    callbacks=[early_stop],
    class_weight=class_weight_dict,   # <-- compensación del desbalance
    verbose=1
)

## 8. Visualización del Entrenamiento

In [ ]:
# Curvas de pérdida y precisión por época
# Train y Val juntas → buen ajuste
# Val sube mientras Train baja → overfitting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curvas de Entrenamiento de la ANN', fontsize=14, fontweight='bold')

epocas = range(1, len(history.history['loss']) + 1)

# Pérdida
axes[0].plot(epocas, history.history['loss'],     'b-o', markersize=3, linewidth=1.5, label='Train Loss')
axes[0].plot(epocas, history.history['val_loss'], 'r-o', markersize=3, linewidth=1.5, label='Val Loss')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida (Categorical Crossentropy)')
axes[0].set_title('Pérdida vs. Épocas')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precisión
axes[1].plot(epocas, history.history['accuracy'],     'b-o', markersize=3, linewidth=1.5, label='Train Accuracy')
axes[1].plot(epocas, history.history['val_accuracy'], 'r-o', markersize=3, linewidth=1.5, label='Val Accuracy')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Precisión')
axes[1].set_title('Precisión vs. Épocas')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Mejor Val Accuracy : {max(history.history["val_accuracy"])*100:.2f}%')
print(f'Épocas entrenadas  : {len(history.history["loss"])}')

## 9. Evaluación del Modelo

In [ ]:
# Evaluación sobre el conjunto de TEST (datos nunca vistos durante el entrenamiento)
loss_test, acc_test = model.evaluate(X_test_sc, y_test_oh, verbose=0)
print(f'Test Loss    : {loss_test:.4f}')
print(f'Test Accuracy: {acc_test*100:.2f}%')

In [ ]:
# predict() devuelve probabilidad para cada clase
# argmax() selecciona la clase con mayor probabilidad
y_pred_prob = model.predict(X_test_sc)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = y_test.values

print('Ejemplos de probabilidades (primeras 5 muestras):')
for i in range(5):
    print(f'  Muestra {i+1}: P(Malo)={y_pred_prob[i,0]:.3f}  '
          f'P(Estándar)={y_pred_prob[i,1]:.3f}  '
          f'P(Bueno)={y_pred_prob[i,2]:.3f}  →  Pred={y_pred[i]}  Real={y_true[i]}')

In [ ]:
# Matriz de confusión: muestra cuántos ejemplos de cada clase real
# fueron asignados a cada clase predicha por el modelo
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Matriz de Confusión - Conjunto de Test', fontsize=14, fontweight='bold')

ConfusionMatrixDisplay(cm, display_labels=['Malo (0)', 'Estándar (1)', 'Bueno (2)']
                      ).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Conteos absolutos')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm.round(2),
                       display_labels=['Malo (0)', 'Estándar (1)', 'Bueno (2)']
                      ).plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Normalizada (por clase real)')

plt.tight_layout()
plt.show()

In [ ]:
# Reporte de clasificación con Precision, Recall y F1-Score por clase
# Precision → de los que predije como X, ¿cuántos son realmente X?
# Recall    → de los que son realmente X, ¿cuántos logré detectar?
# F1-Score  → media armónica entre Precision y Recall
print('=== Reporte de Clasificación ===')
print(classification_report(y_true, y_pred,
                             target_names=['Malo (0)', 'Estándar (1)', 'Bueno (2)']))

## 11. Guardar el Modelo

In [ ]:
# Conectar con drive para guardar el modelo entrenado
from google.colab import drive
drive.mount('/content/drive')
# Guardamos el modelo entrenado para poder reutilizarlo sin reentrenar
model.save('/content/drive/MyDrive/modelo_ANN_multiclass_M2.keras')
print('Modelo guardado en Google Drive como "modelo_ANN_multiclass_M2.keras"')



In [ ]:
#Conectar a drive y guarda el scaler para usarlo en futuras predicciones
from google.colab import drive
drive.mount('/content/drive')
import joblib
joblib.dump(scaler, '/content/drive/MyDrive/scaler_ANN_multiclass_M2.pkl')
print('Scaler guardado en Google Drive como "scaler_ANN_multiclass_M2.pkl"')
